# 01 — Exploratory Data Analysis
**PIX Fraud Intelligence | IBM Portfolio Project**

Dataset: [PIX Fraud BR](https://huggingface.co/datasets/andremessina/pix-fraud-br) — 2M synthetic PIX transactions  
Purpose-built dataset with BCB-aligned modalities and Febraban fraud patterns.

**Goals:**
- Understand class imbalance (fraud rate ~0.77%)
- Explore feature distributions across fraud and legitimate transactions
- Identify key discriminative features: `razao_saldo_residual`, `saldo_anterior_recebedor`
- Generate visualizations for the README

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

os.makedirs('../assets', exist_ok=True)
os.makedirs('../data/raw', exist_ok=True)

## 1. Load Dataset from Hugging Face

In [ ]:
from datasets import load_dataset

print("Loading PIX Fraud BR from Hugging Face...")
ds = load_dataset("andremessina/pix-fraud-br", split="train")
df_full = ds.to_pandas()

print(f"Full dataset: {df_full.shape}")
print(f"Fraud rate:   {df_full['fraude'].mean():.4%}")
df_full.head(3)

In [ ]:
# Sample 100k rows for DB2 Lite (200MB limit)
# Weighted to preserve fraud representation (fraud rate ~0.77%)
df = df_full.sample(n=100_000, random_state=42,
                    weights=df_full['fraude'].map({0: 1, 1: 50}))
df = df.reset_index(drop=True)

# Free 2M-row DataFrame from memory — no longer needed
del df_full

print(f"Sampled: {df.shape}")
print(f"Fraud rate: {df['fraude'].mean():.4%}")

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# sort_index() garante ordem 0=Legítima, 1=Fraude independente do value_counts
counts = df['fraude'].value_counts().sort_index()
axes[0].bar(['Legítima', 'Fraude'], counts.values, color=['#0062ff', '#ff5c49'])
axes[0].set_title('Distribuição de Classes')
axes[0].set_ylabel('Número de transações')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Legítima', 'Fraude'],
            autopct='%1.2f%%', colors=['#0062ff', '#ff5c49'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporção de Classes')

plt.suptitle('Desbalanceamento de Classes — Transações PIX', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 3. Valor da Transação + Fraude por Hora

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#0062ff', 'Legítima'), (1, '#ff5c49', 'Fraude')]:
    axes[0].hist(np.log1p(df[df['fraude'] == label]['valor_brl']),
                 bins=50, alpha=0.6, color=color, label=name)
axes[0].set_title('Distribuição log(valor_brl + 1) por Classe')
axes[0].set_xlabel('log(valor_brl + 1)')
axes[0].legend()

fraud_by_hour = df.groupby('hora_dia')['fraude'].mean()
axes[1].plot(fraud_by_hour.index, fraud_by_hour.values * 100,
             marker='o', color='#ff5c49', linewidth=2)
# hora_dia vai de 0 a 23 — axvspan corrigido para não ultrapassar 23
axes[1].axvspan(20, 23, alpha=0.08, color='red', label='Horário noturno BCB (20h-23h)')
axes[1].axvspan(0, 6, alpha=0.08, color='red')
axes[1].set_title('Taxa de Fraude por Hora do Dia')
axes[1].set_xlabel('Hora'); axes[1].set_ylabel('Taxa de Fraude (%)')
axes[1].set_xticks(range(0, 24)); axes[1].legend()

plt.suptitle('Comportamento das Transações PIX', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/amount_time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 4. Features Discriminativas — Separação por Classe

In [ ]:
NUMERIC_FEATURES = [
    'razao_saldo_residual', 'saldo_anterior_recebedor',
    'saldo_anterior_pagador', 'proporcao_valor_recebedor', 'valor_brl',
]
# Features com escala [0,1] não precisam de log — as demais sim
LOG_FEATURES = {'saldo_anterior_recebedor', 'saldo_anterior_pagador', 'valor_brl'}

fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(18, 5))
for ax, feat in zip(axes, NUMERIC_FEATURES):
    for label, color, name in [(0, '#0062ff', 'Legítima'), (1, '#ff5c49', 'Fraude')]:
        data = df[df['fraude'] == label][feat]
        if feat in LOG_FEATURES:
            data = np.log1p(data)
        ax.hist(data, bins=40, alpha=0.6, color=color, label=name, density=True)
    xlabel = f'log({feat})' if feat in LOG_FEATURES else feat
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_title(feat, fontsize=8, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('Distribuição por Classe — Features Principais', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 5. Modalidade PIX e Taxa de Fraude

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tipo_counts = df['tipo_transacao'].value_counts()
axes[0].bar(tipo_counts.index, tipo_counts.values, color='#0062ff')
axes[0].set_title('Volume por Modalidade PIX')
axes[0].set_ylabel('Transações')
for i, v in enumerate(tipo_counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontsize=9)

fraud_by_tipo = df.groupby('tipo_transacao')['fraude'].mean() * 100
axes[1].bar(fraud_by_tipo.index, fraud_by_tipo.values, color='#ff5c49')
axes[1].set_title('Taxa de Fraude por Modalidade (%)')
axes[1].set_ylabel('Taxa de Fraude (%)')
for i, v in enumerate(fraud_by_tipo.values):
    axes[1].text(i, v + 0.01, f'{v:.2f}%', ha='center', fontsize=9)

plt.suptitle('Modalidades PIX — BCB Manual de Padrões v2.9.0', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/modalities_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 6. Correlation Heatmap

In [ ]:
CORR_FEATURES = [
    'valor_brl', 'saldo_anterior_pagador', 'saldo_posterior_pagador',
    'saldo_anterior_recebedor', 'hora_dia', 'horario_noturno',
    'acima_limite_noturno', 'razao_saldo_residual',
    'proporcao_valor_recebedor', 'fraude',
]

corr = df[CORR_FEATURES].astype(float).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlação — Features Numéricas + Target', fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 7. Save Sampled Dataset

In [ ]:
df.to_parquet('../data/raw/transactions_sampled.parquet', index=False)
print(f"Saved {len(df):,} rows to data/raw/transactions_sampled.parquet")
print(f"Columns: {list(df.columns)}")
print(f"\nSummary:")
print(df.describe().round(2))